In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [2]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for col in df.columns:
        if col.lower() == "species":
            rename_map[col] = "Species"
        elif col.lower() == "originlocation":
            rename_map[col] = "OriginLocation"
    return df.rename(columns=rename_map)


def get_preprocessed_df() -> pd.DataFrame:
    df = pd.read_csv("penguins.csv")
    df = normalize_columns(df)
    null_columns = df.columns[df.isnull().any()]
    numeric_cols = df.select_dtypes(include="number").columns
    for col in null_columns:
        if col in numeric_cols:
            df[col] = df.groupby("Species")[col].transform(lambda x: x.fillna(x.mean()))
    le = LabelEncoder()
    df["OriginLocation"] = le.fit_transform(df["OriginLocation"])
    return df


def split(data: pd.DataFrame):

    encoder = LabelEncoder()
    data["Species"] = encoder.fit_transform(data["Species"])

    # first_class = data[data["Species"] == -1]
    # second_class = data[data["Species"] == 1]
    # first_class= first_class.sample(frac=1).reset_index(drop=True)
    # second_class= second_class.sample(frac=1).reset_index(drop=True)
    # data = pd.concat([first_class, second_class], ignore_index=True)
    
    train_data = data.iloc[np.r_[0:30, 50:80, 100:130]]
    test_data  = data.iloc[np.r_[30:50, 80:100, 130:150]]

    sc = MinMaxScaler()
    train_data = sc.fit_transform(train_data)
    test_data  = sc.transform(test_data) 

    train_data = pd.DataFrame(train_data)
    test_data = pd.DataFrame(test_data)

    X_train = train_data.iloc[:,1:]
    y_train = train_data.iloc[:,0:1]

    X_test = test_data.iloc[:,1:]
    y_test = test_data.iloc[:,0:1]

   

    # Keep original (unscaled) test coords for the scatter plot
    test_original = test_data.copy()

    return X_train, y_train, X_test, y_test,sc


In [ ]:
class Layer:
    def __init__(self, input_num, output_num, bias):
        self.outputs = None
        self.nets = None
        self.errors = None
        self.bias = bias
        self.weights = np.random.randn(input_num + self.bias, output_num) 

class mlp:
    def __init__(self,X_train ,y , hidden_layers, hidden_neurons, learning_rate, epochs, bias, activation_function):
        self.X_train = X_train.copy()
        output_size=len(y.value_counts()) # get the output layer neuron counts 
        input_size =len(X_train.columns) # get the number of the input features
        self.y = y # label column
        self.bias = bias
        self.epochs = epochs
        self.layers = [] # list of layers of the class store each layer nets and y and weights and errors
        self.learning_rate = learning_rate
        self.hidden_layers = hidden_layers # number hidden layers 
        self.layer_num = hidden_layers + 1 # number of layers (hidden + output)
        self.hidden_neurons = hidden_neurons
        self.activation_function = activation_function # 1 for sigmoid 2 for tanh
        
        if bias:
            self.X_train["x0"] = 1 # make a column of ones when bias = 1 to include bias in calculations in the input layer
        
        print(self.X_train.head())
        for i in range(hidden_layers + 1):
            if i == 0: # make the intialization of the input layer
                lyr =Layer(input_size, hidden_neurons, bias)
                self.layers.append(lyr)
                
                # --------------prints for test-------------- 
                # print(f'Layer : {i}')
                # print(lyr.weights)
                
            elif i == hidden_layers: # make the intialization of the output layer
                lyr =Layer(hidden_neurons, output_size, bias)
                self.layers.append(lyr)
                
                # --------------prints for test-------------- 
                # print('Output Layer')
                # print(lyr.weights)
                
            else: # make the intialization of hidden layers
                lyr =Layer(hidden_neurons, hidden_neurons, bias)
                self.layers.append(lyr)
                
                # --------------prints for test-------------- 
                # print(f'Layer : {i}')
                # print(lyr.weights)

    def activationFn(self, actFn, net):
          if (actFn+1) == 1: # Sigmoid Function
              y = 1 / (1 + np.exp(-net))
               
          if (actFn+1) == 2: # Tanh Func
              y = np.tanh(net)
             #y = ((e**net)-(e**(-net))) / ((e**net)+(e**(-net)))
          
          return y
      
    def forward_pass(self ,x):
        for i in range(len(self.layers)): # itterate on each layer 
            if i == 0: # take the input layer and convert it numpy array to be transposable 
                input = np.array(x)
            else: # take the previous output as input for the current layer
                input = self.layers[i-1].outputs
                if self.bias:# if there is a bias add column of ones to the input to match the weights matrix shape 
                    input = np.hstack((input, np.ones((input.shape[0], 1)))) 
            
            net = np.dot(input,self.layers[i].weights)
            self.layers[i].nets = net # store the net matix of the current layer
            
            y = self.activationFn(0,net)
            self.layers[i].outputs = y # store the output matix of the current layer
            
            
            # --------------prints for test-------------- 
            # print(type(self.layers[i].nets))
            # print("layer nets ",i," = ",self.layers[i].nets,"\n")
            # print("layer ys ",i," = ",self.layers[i].outputs)
    
    def backpropagation(self):
        pass  # raneem fn

        
        

In [4]:
df = get_preprocessed_df()

In [5]:
X_train, y_train, X_test, y_test,sc = split(df)

In [6]:
print(len(X_train))

90


In [7]:
model = mlp(X_train,y_train,2,2,0.1,1,0,0)
sum =0
for i in range(len(X_train)):
    sum+=1
    model.forward_pass(X_train.iloc[i,:])
print("the sum = ",sum)    

          1         2         3    4         5
0  0.209205  0.666667  0.155172  1.0  0.250000
1  0.225941  0.511905  0.241379  1.0  0.264706
2  0.259414  0.583333  0.396552  1.0  0.102941
3  0.197677  0.674684  0.251583  1.0  0.234244
4  0.108787  0.738095  0.362069  1.0  0.161765

 input [0.20920502 0.66666667 0.15517241 1.         0.25      ] 


 input [0.96532003 0.13918817] 


 input [0.22594142 0.51190476 0.24137931 1.         0.26470588] 


 input [0.94683859 0.14971648] 


 input [0.25941423 0.58333333 0.39655172 1.         0.10294118] 


 input [0.95433672 0.15150289] 


 input [0.1976774  0.67468416 0.25158339 1.         0.2342437 ] 


 input [0.9637475  0.13751056] 


 input [0.10878661 0.73809524 0.36206897 1.         0.16176471] 


 input [0.96570784 0.1442863 ] 


 input [0.21757322 0.89285714 0.31034483 1.         0.22058824] 


 input [0.9784004  0.11267352] 


 input [0.20083682 0.55952381 0.15517241 1.         0.21323529] 


 input [0.95532248 0.15743378] 


 input [0.